# 3 · LangChain — Agente con framework de orquestación
### Sesión 1 — Agentes de IA, Orquestación y Protocolos

**Complejidad: 🟡 Media**   |   **Dependencias: `anthropic`, `langchain`, `langchain-anthropic`**

Reconstruimos el mismo agente del notebook 2, pero delegando el loop, el parsing de tool calls y el manejo de mensajes a LangChain. La idea es que puedas **comparar directamente** cuánto código te ahorra el framework.

> Si esta instalación te da problemas, los notebooks 1 y 2 (que no dependen de LangChain) siguen funcionando de forma independiente.


## 🎯 Objetivo de aprendizaje

Al terminar este notebook vas a poder:
- Explicar qué problema resuelve un framework de orquestación como LangChain frente a escribir el loop a mano.
- Identificar los conceptos clave de LangChain: Chains, Agents, Tools, Memory y LCEL.
- Reconstruir el mismo agente del notebook anterior usando `AgentExecutor`, y comparar cuánto código se ahorra.


## 📚 Teoría: LangChain como framework de orquestación

Escribir el loop de un agente a mano (como en el notebook 2) funciona bien para ejemplos simples, pero se vuelve difícil de mantener cuando el flujo crece: múltiples herramientas, memoria de conversación, varios agentes colaborando, reintentos ante errores, etc. Un **framework de orquestación** como LangChain existe para resolver justamente eso.

Conceptos clave de LangChain:
- **Chains**: secuencias de pasos (prompt → LLM → parser → siguiente paso) compuestas de forma declarativa.
- **Agents**: deciden dinámicamente qué herramienta usar en cada paso — es la abstracción que reemplaza el loop manual del notebook 2.
- **Tools**: funciones que el agente puede invocar, definidas de forma muy similar al schema que ya usaste (nombre, descripción, parámetros).
- **Memory**: historial de conversación persistido entre llamadas, para que el agente recuerde contexto previo.
- **LCEL** (*LangChain Expression Language*): sintaxis para componer pipelines usando el operador `|`.

**El punto clave de este notebook:** `AgentExecutor` hace **internamente el mismo ciclo ReAct** que construiste a mano — la diferencia es que el framework lo abstrae para que no tengas que reescribirlo cada vez que cambias de proyecto.


## 0. Instalación

In [ ]:
!pip install -q langchain langchain-anthropic langchain-core

### Configurar API key de Anthropic

**Cómo obtenerla:** [console.anthropic.com](https://console.anthropic.com/settings/keys)

Recomendado en Colab: guárdala en **Secrets** (ícono de llave 🔑 a la izquierda) con el nombre `ANTHROPIC_API_KEY` y actívala para este notebook. Si no usas Secrets, te la pedirá por input.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = os.environ.get("ANTHROPIC_API_KEY") or getpass("Pega tu ANTHROPIC_API_KEY: ")

print("API key configurada:", "OK" if os.environ.get("ANTHROPIC_API_KEY") else "FALTA")


## 1. Herramientas (con el decorador `@tool` de LangChain)

In [ ]:
def calculadora(expresion: str) -> str:
    try:
        return str(eval(expresion, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

def buscar_evento_universidad(tema: str) -> str:
    eventos = {
        "ia": "Seminario de Inteligencia Artificial - 25 de julio, Auditorio Principal, 3pm",
        "emprendimiento": "Feria de Emprendimiento - 30 de julio, Plazoleta Central, 9am",
    }
    for k, v in eventos.items():
        if k in tema.lower():
            return v
    return "No se encontraron eventos relacionados con ese tema."


In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

@tool
def calculadora_lc(expresion: str) -> str:
    """Evalúa una expresión matemática simple."""
    return calculadora(expresion)

@tool
def buscar_evento_lc(tema: str) -> str:
    """Busca eventos en el calendario de la universidad por tema."""
    return buscar_evento_universidad(tema)

llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente de la universidad. Usa las herramientas disponibles cuando lo necesites."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

lc_tools = [calculadora_lc, buscar_evento_lc]
agent = create_tool_calling_agent(llm, lc_tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=lc_tools, verbose=True, max_iterations=5)


## 2. Ejecutar el agente

In [ ]:
result = agent_executor.invoke({
    "input": "Busca si hay algún evento de emprendimiento en la universidad y dime en qué lugar es."
})
print("\n=== RESPUESTA FINAL ===")
print(result["output"])


**Nota de comparación:** `AgentExecutor` internamente hace el mismo loop que escribimos a mano en el notebook 2 (`verbose=True` te deja verlo paso a paso). La ventaja de LangChain aparece cuando el flujo crece: memoria multi-turno, múltiples agentes, integración con retrievers, etc.

## 🧪 Ejercicio

Agrega memoria de conversación (`RunnableWithMessageHistory` o `ConversationBufferMemory`) para que el agente recuerde el contexto de una pregunta anterior dentro de la misma sesión.

---
**Siguiente notebook:** `llamaindex_rag.ipynb` — RAG sobre documentos con LlamaIndex.


In [ ]:
# Tu código aquí
